# Genotype–Phenotype Heterogeneity and Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. You will examine clinical, hormone, imaging, and genetic data of three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}\nLicense: {metadata.license}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant FAIR^2 dataset contains several record sets, each with unique `@id` fields. Let's enumerate available record sets and their corresponding fields (if any).

In [ ]:
# List available record sets and their fields

# Some datasets expose record_set ids in metadata.recordSet
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in metadata.recordSet]
else:
    # Attempt automatic discovery via mlcroissant
    for recset in dataset.record_sets():
        record_sets.append(recset['@id'])

print("Available record sets (@id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# List the columns/fields for each record set
for rs_id in record_sets:
    recset_obj = dataset.record_set(rs_id)
    fields = recset_obj['fields'] if 'fields' in recset_obj else []
    print(f"\nRecord set {rs_id} fields (@id):")
    for field in fields:
        print(f"  - {field['@id']} ({field.get('name', '')})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

Use the record set and field `@id`s from the overview section.

In [ ]:
# Extract data from each record set
dataframes = {}

# Choose the record set IDs discovered earlier
# If not available, manually set up the example record set IDs based on dataset description
# Example placeholder recordSet IDs (replace with actual IDs or those returned above):

example_record_sets = record_sets if record_sets else [
    'table1', # Clinical features
    'table2', # Hormone and imaging
    'table3'  # Genetic variants
]

for record_set in example_record_sets:
    print(f"\nLoading records for record set @id: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        print(f"  Loaded {len(records)} records.")
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample data:")
        print(df.head())
    except Exception as e:
        print(f"  Error loading record set {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We will demonstrate these on one record set (e.g., clinical features or hormone levels), referencing fields by their `@id`.

In [ ]:
# Example: Perform EDA on Table 2 (hormone and imaging)
# Choose a record set ID from previous output (e.g., 'table2')
record_set_id = example_record_sets[1] if len(example_record_sets) > 1 else None
df = dataframes.get(record_set_id, pd.DataFrame())
if df.empty:
    print(f"No data found for record set @id: {record_set_id}")
else:
    print(f"EDA for record set @id: {record_set_id}")
    # List fields for this record set
    recset_obj = dataset.record_set(record_set_id)
    fields = recset_obj['fields'] if 'fields' in recset_obj else []
    # Find numeric field (for demonstration, pick hormone level fields)
    numeric_field_id = None
    for field in fields:
        # Assume fields have 'dataType' and '@id'; choose a Float or Integer field
        if field.get('dataType') in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
            numeric_field_id = field['@id']
            break
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a categorical field for summary
        group_field_id = None
        for field in fields:
            if field.get('dataType') == 'schema:Text' and field['@id'] in df.columns:
                group_field_id = field['@id']
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for demonstration.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below is an example visualization of hormone levels for the patients.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution for Table 2
if record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Optional: Visualize by group field
    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

This notebook demonstrated how to load, explore, process, and visualize data from the FAIR^2 Croissant dataset using the `mlcroissant` library.

- The dataset offers structured clinical, hormone, imaging, and genetic variant information on NC-21OHD-associated infertility in a small cohort.
- Data was referenced and loaded by unique `@id` for record sets and fields.
- Exploratory analysis highlighted key workflow steps from record set overview to field-level visualization.

Due to the limited cohort (N=3), findings are illustrative for clinical research, but not suitable for machine learning model development.